In [ ]:
# Different Method for solving overfitting problem
  # Adding more data
  # Reduce the complexity of a Model
  # Regularization
  # Dropouts
  # Data augmentation
  # Batch Normalization
  # Early Stoping

In [ ]:
# Dropout
  # Applied to the hidden layer
  # Execute after ReLU Function
  # Randomely turn off p% neron of hidden layer
  # It has regularization effect
  # During evaluation dropout not use

In [ ]:
# Batch Normalization
  # applied to the hidden layer
  # Applied after linear layer anf before activation function
  # Include Learnable parameters
  # It has regularization parameters

In [ ]:
# L-2 Regularization
  # Applied to the model weights
  # Introduced via model loss function
  # Reduce overfitting

In [1]:
# Iport Required Libraries
import pandas as pd
from sklearn.model_selection import train_test_split
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torch.optim as optim
import matplotlib.pyplot as plt

In [2]:
# Set random seeds for reproducibility
torch.manual_seed(42)

In [3]:
# Check GPU availability
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Available Device is: {device}')

Available Device is: cuda


In [4]:
df = pd.read_csv('/content/fashion-mnist_train.csv')
df.head()

,label,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,pixel9,...,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783,pixel784
0,2,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,9,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,6,0,0,0,0,0,0,0,5,0,...,0,0,0,30,43,0,0,0,0,0
3,0,0,0,0,1,2,0,0,0,0,...,3,0,0,0,0,1,0,0,0,0
4,3,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [8]:
# Extract Features values and Target values
X=df.iloc[:, 1:].to_numpy()
y=df.iloc[:, 0].to_numpy()

In [9]:
# Split the Dataset into train and testset
X_train,X_test, y_train,y_test=train_test_split(
    X,y, test_size=0.2, random_state=42
)

In [10]:
# Scaling the features
X_train=X_train/255.0
X_test=X_test/255.0

In [11]:
# Create CustomDataset Class
class CustomDataset(Dataset):

    def __init__(self, features, labels):
        self.features=torch.tensor(features, dtype=torch.float32)
        self.labels=torch.tensor(labels, dtype=torch.long)
    def __len__(self):
        return len(self.features)
    def __getitem__(self, index):
        return self.features[index], self.labels[index]

In [12]:
# Create train dataset object
train_dataset=CustomDataset(X_train, y_train)

In [13]:
# Create test Dataset object
test_dataset=CustomDataset(X_test, y_test)

In [14]:
# Create train and test dataloader
# Add pin_memory to speed the training process
train_loader=DataLoader(train_dataset, batch_size=32, shuffle=True, pin_memory=True)
test_loader=DataLoader(test_dataset, batch_size=32, shuffle=False, pin_memory=True)

In [18]:
# Define Neural Network Class

class MyNN(nn.Module):

    def __init__(self, num_features):
        super().__init__()
        self.model=nn.Sequential(
            nn.Linear(num_features, 128),
            nn.BatchNorm1d(128),        # add Batchnormalization
            nn.ReLU(),
            nn.Dropout(p=0.3),          # add dropout
            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(p=0.3),
            nn.Linear(64, 10)
        )
    def forward(self, x):
        return self.model(x)

In [16]:
# Set Learning Rate and epochs
epochs=100
learning_rate=0.1

In [19]:
# Initiate the model
model=MyNN(X_train.shape[1])
# Move model to GPU
model=model.to(device)

# Loss Function
criterion=nn.CrossEntropyLoss()

# Optimizer
optimizer=optim.SGD(model.parameters(), lr=learning_rate, weight_decay=0.00001)   # Add L-2 Regularization

In [20]:
# training loop

for epoch in range(epochs):

  total_epoch_loss = 0

  for batch_features, batch_labels in train_loader:
    # Move data to GPU
    batch_features=batch_features.to(device)
    batch_labels=batch_labels.to(device)

    # forward pass
    outputs = model(batch_features)

    # calculate loss
    loss = criterion(outputs, batch_labels)

    # back pass
    optimizer.zero_grad()
    loss.backward()

    # update grads
    optimizer.step()

    total_epoch_loss = total_epoch_loss + loss.item()

  avg_loss = total_epoch_loss/len(train_loader)
  print(f'Epoch: {epoch + 1} , Loss: {avg_loss}')

Epoch: 1 , Loss: 0.6268073388636112
Epoch: 2 , Loss: 0.49891244746247926
Epoch: 3 , Loss: 0.4555881256212791
Epoch: 4 , Loss: 0.43305151361227034
Epoch: 5 , Loss: 0.4191096511433522
Epoch: 6 , Loss: 0.4076412619451682
Epoch: 7 , Loss: 0.39487797682980696
Epoch: 8 , Loss: 0.384124314631025
Epoch: 9 , Loss: 0.3753511316925287
Epoch: 10 , Loss: 0.36715082200368243
Epoch: 11 , Loss: 0.35952148493131003
Epoch: 12 , Loss: 0.35457769395411015
Epoch: 13 , Loss: 0.3508200279722611
Epoch: 14 , Loss: 0.34208862846593063
Epoch: 15 , Loss: 0.3401680876612663
Epoch: 16 , Loss: 0.3353192442258199
Epoch: 17 , Loss: 0.3323983087937037
Epoch: 18 , Loss: 0.33111225778857867
Epoch: 19 , Loss: 0.3263209560016791
Epoch: 20 , Loss: 0.319715834227701
Epoch: 21 , Loss: 0.3206097944478194
Epoch: 22 , Loss: 0.3146722544009487
Epoch: 23 , Loss: 0.3150818480700254
Epoch: 24 , Loss: 0.3109493556320667
Epoch: 25 , Loss: 0.3083593556334575
Epoch: 26 , Loss: 0.3101441786984603
Epoch: 27 , Loss: 0.3045185763463378
Epoc

In [25]:
# evaluation code
total=0
correct=0
with torch.no_grad():
  for batch_features, batch_labels in test_loader:
    # Move to GPU
    batch_features=batch_features.to(device)
    batch_labels=batch_labels.to(device)
    output=model(batch_features)
    _, predicted=torch.max(output.data, 1)
    total=total+batch_labels.shape[0]
    correct=correct+(predicted==batch_labels).sum().item()
print(f'Accuracy: {100*correct/total}')


Accuracy: 85.775


In [23]:
# evaluation code
total=0
correct=0
with torch.no_grad():
  for batch_features, batch_labels in train_loader:
    # Move to GPU
    batch_features=batch_features.to(device)
    batch_labels=batch_labels.to(device)
    output=model(batch_features)
    _, predicted=torch.max(output.data, 1)
    total=total+batch_labels.shape[0]
    correct=correct+(predicted==batch_labels).sum().item()
print(f'Accuracy: {100*correct/total}')

Accuracy: 91.47083333333333


In [ ]:
# Decrease the accuracy difference between test and train dataset after applying optimization